## NOTEBOOK 2

### Purpose of this notebook is to generate multitple synthetic musicXML with MIDI files for eligable pieces


### Download n(ASAP) dataset if it doesn't exist inside the data folder

In [29]:
import os, sys, json, random, copy, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET

sys.path.insert(0, str(Path.cwd().parent))
from utils.constants import DATASET_LOCATION, ARTIFACTS_LOCATION

if not (DATASET_LOCATION).exists():
    import subprocess
    subprocess.run(['git', 'clone',  '--depth', '1', 'https://github.com/CPJKU/asap-dataset.git', str(DATASET_LOCATION)], check=True)
    
ARTIFACTS_LOCATION.mkdir(exist_ok=True, parents=True)
print('Drive base :', ARTIFACTS_LOCATION)
print('ASAP root  :', DATASET_LOCATION, '| exists:', DATASET_LOCATION.exists())




Drive base : D:\Projects\fyp\artifacts
ASAP root  : D:\Projects\fyp\data\asap-dataset | exists: True


### 

In [30]:
from utils.constants import LEVEL_ORDER
from music21 import converter

def oneLevelShift(marking, rng):
    if marking not in LEVEL_ORDER:
        return None
    idx = LEVEL_ORDER.index(marking)
    options = [j for j in (idx -1, idx + 1) if j >= 0 and j < len(LEVEL_ORDER)]
    return LEVEL_ORDER[rng.choice(options)] if options else None

def scanDynamics(path):
    if not path.exists():
        return None
    tree = ET.parse(path)
    root = tree.getroot()
    constants , wedges = {} , {"crescendo": 0, "diminuendo": 0, "stop": 0}

    for dynamics in root.iter('dynamics'):
        for child in dynamics:
            constants[child.tag] = constants.get(child.tag, 0) + 1

    for wedge in root.iter('wedge'):
        wedgeType = wedge.get('type', "")
        if wedgeType in wedges:
            wedges[wedgeType] += 1

    return constants, wedges

def checkWedgeBalance(path):
    tree = ET.parse(path)
    root = tree.getroot()
    issues = []
    for part in root.iter('part'):
        id = part.get('id', "")
        openStacks = {}

        for wedge in part.iter('wedge'):
            num = wedge.get('number', "1")
            wtype = wedge.get('type', "")
            if wtype in ("crescendo", "diminuendo"):
                if openStacks.get(num, 0) > 0:
                    issues.append(f"Unclosed wedge {wtype} in part {id} (number={num})")
                openStacks[num] = openStacks.get(num, 0) + 1
            elif wtype == "stop":
                if openStacks.get(num, 0) == 0:
                    issues.append(f"Stop wedge without open in part {id} (number={num})")
                else:
                    openStacks[num] -= 1
        for num, count in openStacks.items():
            if count > 0:
                issues.append(f"Unclosed wedge in part {id} (number={num})")
    return len(issues) == 0, issues

def validatePair(originalPath, syntheticPath, changelog, checkNotes = True):
    results = {}
    if checkNotes:
        original = converter.parse(originalPath)
        synthetic = converter.parse(syntheticPath)
        originalNotes = []
        syntheticNotes = []

        for note in original.recurse().notes :
            if note.isNote:
                originalNotes.append((note.pitch.nameWithOctave, float(note.offset), float(note.quarterLength)))
        for note in synthetic.recurse().notes :
            if note.isNote:
                syntheticNotes.append((note.pitch.nameWithOctave, float(note.offset), float(note.quarterLength)))

        results["notes_identical"] = {
                    "passed": originalNotes == syntheticNotes,
                    "detail": f"original={len(originalNotes)} notes, synthetic={len(syntheticNotes)} notes"}

    originalConstants, originalWedges = scanDynamics(originalPath)
    syntheticConstants, syntheticWedges = scanDynamics(syntheticPath)

    results["dynamics_constants_status"] = {
        "passed" : sum(originalConstants.values()) == sum(syntheticConstants.values()),
        "detail" : f"original={sum(originalConstants.values())}, synthetic={sum(syntheticConstants.values())}"}

    results["dynamics_wedges_status"] = {
        "passed": originalWedges['crescendo'] + originalWedges['diminuendo'] == syntheticWedges['crescendo'] + syntheticWedges['diminuendo']
        and originalWedges['stop'] == syntheticWedges['stop'],
        "detail": f"original={originalWedges}, synthetic={syntheticWedges}"
    }

    expected = dict(originalConstants)
    expectedWedge = dict(originalWedges)
    for change in changelog:
        if change['type'] == 'constant':
            expected[change["from"]] -= 1
            expected[change["to"]] = expected.get(change["to"], 0) + 1
        elif change['type'] == 'wedge':
            expectedWedge[change["from"]] -= 1
            expectedWedge[change["to"]] += 1

    expected = {key: v for key, v in expected.items() if v > 0}

    results["changelog_applied_exactly"] = {
            "passed": expected == syntheticConstants and expectedWedge == syntheticWedges,
            "detail": f"constants expected={expected}, got={syntheticConstants}; wedges expected={expectedWedge}, got={syntheticWedges}"}

    results["at_least_one_corruption"] = {
            "passed": len(changelog) > 0,
            "detail": f"{len(changelog)} markings corrupted"}

    okOriginal, issuesOriginal = checkWedgeBalance(originalPath)
    okSynthetic, issuesSynthetic = checkWedgeBalance(syntheticPath)
    results["wedge_structure_not_degraded"] = {
            "passed": issuesSynthetic == issuesOriginal,
            "detail": ("identical structure"
                       + ("" if okOriginal else f" (pre-existing issues in original: {len(issuesOriginal)})"))
                      if issuesSynthetic == issuesOriginal else
                      f"NEW issues: {'; '.join(sorted(set(issuesSynthetic) - set(issuesOriginal))[:3])}"}
    return results


In [31]:
from utils.constants import CONSTANT_INVERSION, WEDGE_INVERSION

def generateSyntheticXML(inputPath, outputPath, seed=42, mode="shift", wedgeFlipProb= 0.5):
    if (mode not in ("shift", "invert")):
        raise ValueError(f"Invalid mode: {mode}. Must be 'shift' or 'invert'.")

    rng = random.Random(seed)
    tree = ET.parse(inputPath)
    root = tree.getroot()
    changelog = []

    for dynamics in root.iter('dynamics'):
        for child in dynamics:
            old = child.tag
            if mode == "invert":
                new = CONSTANT_INVERSION.get(old)
            else :
                new = oneLevelShift(old, rng)
            if new is not None and new != old:
                child.tag = new
                changelog.append({"type": "constant", "from": old, "to": new})

    for wedge in root.iter("wedge"):
        old = wedge.get("type", "")
        new = WEDGE_INVERSION.get(old)

        if not new:
            continue
        if mode == "invert":
            wedge.set("type", new)
            changelog.append({"type": "wedge", "from": old, "to": new})
        elif rng.random() < wedgeFlipProb:
            wedge.set("type", new)
            changelog.append({"type": "wedge", "from": old, "to": new})

    Path(outputPath).parent.mkdir(parents=True, exist_ok=True)
    tree.write(outputPath, encoding="UTF-8", xml_declaration=True)
    return changelog

In [32]:
from utils.constants import ARTIFACTS_LOCATION, SYNTH_ROOT, CAP, SYNTH_RATIO, FULL_INVERSION_SHARE, SEED

metadataPath = ARTIFACTS_LOCATION / 'cleaned_metadata.csv'
if not metadataPath.exists():
    raise FileNotFoundError(f'{metadataPath} not found. Please run "Notebook 1 Setup.ipynb" again to generate it.')

data = pd.read_csv(metadataPath)
realCount = data.groupby('folder').size()
softCapPerformances = realCount[realCount > 13]

dataframe = data.drop_duplicates('folder')[['composer', 'folder', 'xml_score', 'midi_score']].reset_index(drop=True)
SYNTH_ROOT.mkdir(parents=True, exist_ok=True)
dataframe.head(4)

planRows = []
for index, row in dataframe.iterrows():
    real = realCount[row.folder]
    cappedReal = min(real, CAP)
    numberOfSynthPieces = max(1, round(cappedReal * SYNTH_RATIO ))
    
    if numberOfSynthPieces == 1:
        numOfInvert = 0
    else: 
        numOfInvert = min(numberOfSynthPieces - 1, max(1, round(numberOfSynthPieces * FULL_INVERSION_SHARE)))
        
    modes = ['shift'] * (numberOfSynthPieces - numOfInvert) + ['invert'] * numOfInvert
    
    for idx, mode in enumerate(modes):
        planRows.append({"piece": row.folder, "composer" : row.composer, "xmlScore": row.xml_score, 'midi_score': row.midi_score, 
                         'index': idx, 'mode': mode,
                         'seed': SEED + index * 100 + idx, 
                         'real_count': real, 'capped_real': cappedReal})
    
plan = pd.DataFrame(planRows)
display(plan.head(4))
print(f'{plan.piece.nunique()} pieces -> {len(plan)} synthetic examples planned')
print(plan['mode'].value_counts())
print('examples per piece:', plan.groupby('piece').size().value_counts().sort_index().to_dict())
    




,piece,composer,xmlScore,midi_score,index,mode,seed,real_count,capped_real
0,Bach/Fugue/bwv_846,Bach,Bach/Fugue/bwv_846/xml_score.musicxml,Bach/Fugue/bwv_846/midi_score.mid,0,shift,42,1,1
1,Bach/Fugue/bwv_848,Bach,Bach/Fugue/bwv_848/xml_score.musicxml,Bach/Fugue/bwv_848/midi_score.mid,0,shift,142,9,9
2,Bach/Fugue/bwv_848,Bach,Bach/Fugue/bwv_848/xml_score.musicxml,Bach/Fugue/bwv_848/midi_score.mid,1,shift,143,9,9
3,Bach/Fugue/bwv_848,Bach,Bach/Fugue/bwv_848/xml_score.musicxml,Bach/Fugue/bwv_848/midi_score.mid,2,invert,144,9,9


187 pieces -> 286 synthetic examples planned
mode
shift     231
invert     55
Name: count, dtype: int64
examples per piece: {1: 132, 2: 26, 3: 14, 4: 15}


In [33]:
hasDyn = {}

for row in dataframe.itertuples(index=False):
    try:
     oc, ow = scanDynamics(DATASET_LOCATION / row.xml_score)
     hasDyn[row.folder] = bool(oc) or ow['crescendo'] > 0 or ow['diminuendo'] > 0
    except:
        hasDyn[row.folder] = False

nWith = sum(hasDyn.values())
print(f'{nWith} / {len(dataframe)} selected pieces have dynamics '
      f'({len(dataframe) - nWith} will be skipped - typically Bach urtext)')
planSel = plan[plan.piece.isin([p for p, ok in hasDyn.items() if ok])]
print(f'-> {len(planSel)} synthetic examples will be generated')
planSel.head(4)

128 / 187 selected pieces have dynamics (59 will be skipped - typically Bach urtext)
-> 212 synthetic examples will be generated


,piece,composer,xmlScore,midi_score,index,mode,seed,real_count,capped_real
74,Balakirev/Islamey,Balakirev,Balakirev/Islamey/xml_score.musicxml,Balakirev/Islamey/midi_score.mid,0,shift,5942,10,10
75,Balakirev/Islamey,Balakirev,Balakirev/Islamey/xml_score.musicxml,Balakirev/Islamey/midi_score.mid,1,shift,5943,10,10
76,Balakirev/Islamey,Balakirev,Balakirev/Islamey/xml_score.musicxml,Balakirev/Islamey/midi_score.mid,2,invert,5944,10,10
77,Beethoven/Piano_Sonatas/1-1,Beethoven,Beethoven/Piano_Sonatas/1-1/xml_score.musicxml,Beethoven/Piano_Sonatas/1-1/midi_score.mid,0,shift,6042,1,1


In [34]:
from utils.constants import SYNTHETIC_MIDI_OUTPUT_LOCATION, SYNTHETIC_XML_OUTPUT_LOCATION, DATASET_LOCATION, WEDGE_FLIP_PROB

logs , values = [] , []
current = time.time()
for idx, row in enumerate(planSel.itertuples(index= False),1):
    stem = f'synthetic_{row.index:02d}_{row.mode}'
    xmlDestination = SYNTHETIC_XML_OUTPUT_LOCATION / row.piece / f'{stem}.musicxml'
    midiDestination = SYNTHETIC_MIDI_OUTPUT_LOCATION / row.piece / f'{stem}.mid'
    manifestDestination = SYNTHETIC_MIDI_OUTPUT_LOCATION / row.piece/ f'{stem}_manifest.json'

    rec = { 'piece' : row.piece, 'idx' : row.index, 'mode': row.mode, 'seed': row.seed}

    try:
        changeLog = generateSyntheticXML(DATASET_LOCATION / row.xmlScore, xmlDestination,
                                         seed=row.seed, mode=row.mode,
                                         wedgeFlipProb=WEDGE_FLIP_PROB
                                         )

        if not changeLog:
            rec['status'] = 'nothing corruptible'
            logs.append(rec)
            continue
        manifestDestination.parent.mkdir(parents=True, exist_ok=True)
        manifestDestination.write_text(json.dumps(
                    {'piece': row.piece, 'mode': row.mode, 'seed': row.seed,
                     'changes': changeLog}, indent=1))

        res = validatePair(DATASET_LOCATION/ row.xmlScore, xmlDestination, changeLog,checkNotes=(row.index == 0))

        allres = {**{f'xml_{k}': v for k, v in res.items()}}

        vrow = {'piece': row.piece, 'k': row.index, 'mode': row.mode, **{k: v['passed'] for k, v in allres.items()}}
        vrow['all_passed'] = all(v['passed'] for v in allres.values())
        values.append(vrow)
        rec.update(status='generated', n_corruptions=len(changeLog),
                        #    notes_rendered=meta['notes_changed'],
                        #    overlaps_removed=len(meta['dropped_wedges'])
        )

    except Exception as e:
         rec['status'] = f'error: {type(e).__name__}: {e}'
    logs.append(rec)
    if idx % 10 == 0:
        print(f'{idx}/{len(planSel)} examples | {time.time()-current:.0f}s elapsed')

gen_df, val_df = pd.DataFrame(logs), pd.DataFrame(values)
print()
print(gen_df['status'].value_counts())

gen_df.to_csv(SYNTH_ROOT / 'generation_log.csv', index=False)
val_df.to_csv(SYNTH_ROOT / 'validation_log.csv', index=False)

d:\Projects\fyp\.venv\Lib\site-packages\music21\musicxml\xmlToM21.py:2075: MusicXMLWarning: Warning: measure 133 in part Pianois overfull: 521/160 > 3.0,assuming 3.0 is correct.
  self.adjustTimeAttributesFromMeasure(m)
d:\Projects\fyp\.venv\Lib\site-packages\music21\musicxml\xmlToM21.py:2075: MusicXMLWarning: Warning: measure 135 in part Pianois overfull: 521/160 > 3.0,assuming 3.0 is correct.
  self.adjustTimeAttributesFromMeasure(m)
d:\Projects\fyp\.venv\Lib\site-packages\music21\musicxml\xmlToM21.py:2075: MusicXMLWarning: Warning: measure 137 in part Pianois overfull: 521/160 > 3.0,assuming 3.0 is correct.
  self.adjustTimeAttributesFromMeasure(m)
d:\Projects\fyp\.venv\Lib\site-packages\music21\musicxml\xmlToM21.py:2075: MusicXMLWarning: Warning: measure 25 in part Pianois overfull: 241/160 > 1.5,assuming 1.5 is correct.
  self.adjustTimeAttributesFromMeasure(m)
d:\Projects\fyp\.venv\Lib\site-packages\music21\musicxml\xmlToM21.py:2075: MusicXMLWarning: Warning: measure 60 in part P

10/212 examples | 29s elapsed
20/212 examples | 52s elapsed
30/212 examples | 83s elapsed


d:\Projects\fyp\.venv\Lib\site-packages\music21\musicxml\xmlToM21.py:5542: MusicXMLWarning: Could not import pedal: Error in getting PedalMark
  self.setDirectionInDirectionType(mxSpecificDirectionTag,


40/212 examples | 103s elapsed
50/212 examples | 118s elapsed
60/212 examples | 137s elapsed


d:\Projects\fyp\.venv\Lib\site-packages\music21\musicxml\xmlToM21.py:2075: MusicXMLWarning: Warning: measure 75 in part Pianois overfull: 289/96 > 3.0,assuming 3.0 is correct.
  self.adjustTimeAttributesFromMeasure(m)
d:\Projects\fyp\.venv\Lib\site-packages\music21\musicxml\xmlToM21.py:2075: MusicXMLWarning: Warning: measure 252 in part Pianois overfull: 289/96 > 3.0,assuming 3.0 is correct.
  self.adjustTimeAttributesFromMeasure(m)


...

160/212 examples | 297s elapsed


d:\Projects\fyp\.venv\Lib\site-packages\music21\musicxml\xmlToM21.py:2075: MusicXMLWarning: Warning: measure 297 in part Pianois overfull: 289/48 > 6.0,assuming 6.0 is correct.
  self.adjustTimeAttributesFromMeasure(m)
d:\Projects\fyp\.venv\Lib\site-packages\music21\musicxml\xmlToM21.py:5542: MusicXMLWarning: Could not import octave-shift: Error in getting Ottava
  self.setDirectionInDirectionType(mxSpecificDirectionTag,
d:\Projects\fyp\.venv\Lib\site-packages\music21\musicxml\xmlToM21.py:2075: MusicXMLWarning: Warning: measure 217 in part Pianois overfull: 241/120 > 2.0,assuming 2.0 is correct.
  self.adjustTimeAttributesFromMeasure(m)
d:\Projects\fyp\.venv\Lib\site-packages\music21\musicxml\xmlToM21.py:2075: MusicXMLWarning: Warning: measure 218 in part Pianois overfull: 329/160 > 2.0,assuming 2.0 is correct.
  self.adjustTimeAttributesFromMeasure(m)
d:\Projects\fyp\.venv\Lib\site-packages\music21\musicxml\xmlToM21.py:2075: MusicXMLWarning: Warning: measure 217 in part Pianois overfu

170/212 examples | 323s elapsed
180/212 examples | 342s elapsed
190/212 examples | 362s elapsed
200/212 examples | 378s elapsed


d:\Projects\fyp\.venv\Lib\site-packages\music21\musicxml\xmlToM21.py:2075: MusicXMLWarning: Warning: measure 31 in part Midi_1is overfull: 1153/480 > 2.0,assuming 2.0 is correct.
  self.adjustTimeAttributesFromMeasure(m)
d:\Projects\fyp\.venv\Lib\site-packages\music21\musicxml\xmlToM21.py:2075: MusicXMLWarning: Warning: measure 32 in part Midi_1is overfull: 1153/480 > 2.0,assuming 2.0 is correct.
  self.adjustTimeAttributesFromMeasure(m)
d:\Projects\fyp\.venv\Lib\site-packages\music21\musicxml\xmlToM21.py:2075: MusicXMLWarning: Warning: measure 33 in part Midi_1is overfull: 1153/480 > 2.0,assuming 2.0 is correct.
  self.adjustTimeAttributesFromMeasure(m)
d:\Projects\fyp\.venv\Lib\site-packages\music21\musicxml\xmlToM21.py:2075: MusicXMLWarning: Warning: measure 34 in part Midi_1is overfull: 1153/480 > 2.0,assuming 2.0 is correct.
  self.adjustTimeAttributesFromMeasure(m)
d:\Projects\fyp\.venv\Lib\site-packages\music21\musicxml\xmlToM21.py:2075: MusicXMLWarning: Warning: measure 35 in p

210/212 examples | 399s elapsed

status
generated    212
Name: count, dtype: int64
